In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Install Required Libraries
# Run this cell first, every session
# ─────────────────────────────────────────────

%pip install google-generativeai

StatementMeta(, 83995e48-97f3-4db5-9374-620a0f9410ab, 8, Finished, Available, Finished, False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 9.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.1/173.1 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.3/252.3 kB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 156.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Setup
# ─────────────────────────────────────────────



import google.generativeai as genai
import pandas as pd
import time

GOOGLE_API_KEY = "YOUR_API_KEY"  # paste your Gemini key
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('models/gemini-2.5-flash')

print("✅ Gemini client initialized")

StatementMeta(, 83995e48-97f3-4db5-9374-620a0f9410ab, 10, Finished, Available, Finished, False)

/tmp/ipykernel_7223/2778543864.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — Load & Join Data Directly From Lakehouse
# Recreates vw_HR_Master logic without going 
# through the Warehouse (Spark can't read Warehouse views)
# ─────────────────────────────────────────────

df_gold = spark.sql("""
    SELECT
        s.EmployeeID,
        s.Name,
        s.Department,
        s.JobRole,
        s.Age,
        s.YearsAtCompany,
        s.MonthlyIncome,
        s.YearsSinceLastPromotion,
        s.OverTime,
        s.JobSatisfaction,
        s.WorkLifeBalance,
        s.EnvironmentSatisfaction,
        s.RelationshipSatisfaction,
        s.RiskLevel,
        s.MaritalStatus,
        s.Attrition,
        m.ML_Risk_Score_Pct,
        m.ClusterName
    FROM TechNova_HR_Lakehouse.dbo.silver_master_employees s
    LEFT JOIN TechNova_HR_Lakehouse.dbo.ml_attrition_predictions m
        ON s.EmployeeID = m.EmployeeID
    WHERE s.Attrition = 'No'
    ORDER BY m.ML_Risk_Score_Pct DESC
""").toPandas()

print(f"✅ Loaded {len(df_gold)} current employees")
print(df_gold[['Name', 'Department', 'ML_Risk_Score_Pct']].head(3))

StatementMeta(, 83995e48-97f3-4db5-9374-620a0f9410ab, 11, Finished, Available, Finished, False)

✅ Loaded 2335 current employees
            Name         Department  ML_Risk_Score_Pct
0    Abdul Basak  IT Infrastructure               70.5
1    Hema Mitter    Human Resources               69.3
2  Sudiksha Kari  IT Infrastructure               68.7


In [5]:
import google.generativeai as genai

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

StatementMeta(, 83995e48-97f3-4db5-9374-620a0f9410ab, 13, Finished, Available, Finished, False)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [4]:
# ─────────────────────────────────────────────
# CELL 4 — Prompt Builder + API Call
# ─────────────────────────────────────────────
import pandas as pd
import time
def build_prompt(employee: dict) -> str:
    return f"""You are an expert HR Business Partner AI assistant 
at TechNova Solutions. Analyze the following employee risk data 
and generate a professional, empathetic, and actionable HR 
intervention recommendation. Be specific, concise, and use 
professional HR language.

Employee Profile:
- Name: {employee['Name']}
- Department: {employee['Department']}
- Role: {employee['JobRole']}
- Age: {employee['Age']}
- Years At Company: {employee['YearsAtCompany']}
- Monthly Income: ₹{employee['MonthlyIncome']:,}
- Years Since Last Promotion: {employee['YearsSinceLastPromotion']}
- Overtime: {employee['OverTime']}
- Job Satisfaction: {employee['JobSatisfaction']}/4
- Work Life Balance: {employee['WorkLifeBalance']}/4
- Environment Satisfaction: {employee['EnvironmentSatisfaction']}/4
- Relationship Satisfaction: {employee['RelationshipSatisfaction']}/4
- ML Attrition Risk Score: {employee['ML_Risk_Score_Pct']}%
- Risk Level: {employee['RiskLevel']}
- Employee Segment: {employee['ClusterName']}
- Marital Status: {employee['MaritalStatus']}

Generate your response in EXACTLY this format, Do not use Markdown formatting (no **, ##, or *). 
Use plain text only, with clear section headers in ALL CAPS 
followed by a colon, and numbered lists using plain numbers.:

Risk Summary
[2 sentences: mention employee name, tenure, and ML risk score.
State urgency clearly.]

Key Risk Drivers
[3-5 bullet points identifying specific factors
driving attrition risk for THIS employee.
Be specific — reference actual scores from the data.]

Recommended Interventions
[Exactly 3 actions with specific timelines:
1. [Action] — Complete within [X days/weeks]
2. [Action] — Complete within [X days/weeks]
3. [Action] — Complete within [X days/weeks]]

Manager Advisory
[1 sentence: key lesson or policy recommendation
based on this employee's situation.]"""


def generate_risk_card(employee: dict) -> str:
    try:
        response = model.generate_content(build_prompt(employee))
        return response.text
    except Exception as e:
        return f"Error generating risk card: {str(e)}"


# ── Generate for top 50 highest risk employees ──
TOP_N = 15
top_risk = df_gold.head(TOP_N).copy()

print(f"Generating AI risk cards for top {TOP_N} at-risk employees...")
print(f"Expected time: ~{TOP_N * 2} seconds\n")

risk_cards = []

for idx, employee in top_risk.iterrows():
    print(f"Processing {idx+1}/{TOP_N}: {employee['Name']}...")
    
    card_text = generate_risk_card(employee.to_dict())
    
    risk_cards.append({
        'EmployeeID':    employee['EmployeeID'],
        'Name':          employee['Name'],
        'Department':    employee['Department'],
        'ML_Risk_Score': employee['ML_Risk_Score_Pct'],
        'RiskLevel':     employee['RiskLevel'],
        'AI_Risk_Card':  card_text,
        'Generated_At':  pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
    })
    
    time.sleep(1)

df_risk_cards = pd.DataFrame(risk_cards)
print(f"\n✅ Generated {len(df_risk_cards)} AI risk cards")
print("\nSample card:")
print(df_risk_cards.iloc[0]['AI_Risk_Card'])

StatementMeta(, 83995e48-97f3-4db5-9374-620a0f9410ab, 12, Finished, Available, Finished, False)

Generating AI risk cards for top 50 at-risk employees...
Expected time: ~100 seconds

Processing 1/50: Abdul Basak...
Processing 2/50: Hema Mitter...
Processing 3/50: Sudiksha Kari...
Processing 4/50: Faris Sant...
Processing 5/50: Odika Garde...
Processing 6/50: Ekalinga Dua...
Processing 7/50: Henry Jain...
Processing 8/50: Ishanvi Kohli...
Processing 9/50: Irya Kata...
Processing 10/50: Gunbir Datta...
Processing 11/50: Patrick Sahni...
Processing 12/50: Advay Prakash...
Processing 13/50: Indira Kibe...
Processing 14/50: Sneha Chaudhari...
Processing 15/50: Netra Manda...
Processing 16/50: Aashi Rajagopal...
Processing 17/50: Nathaniel Bhatti...
Processing 18/50: Keya Varma...
Processing 19/50: Kabir Mitra...
Processing 20/50: Geetika Saini...
Processing 21/50: Manya Mukhopadhyay...
Processing 22/50: Zayan Mall...
Processing 23/50: Yutika Samra...
Processing 24/50: Pavani Palan...
Processing 25/50: Henry Chada...
Processing 26/50: Gagan Prabhakar...
Processing 27/50: Aachal Borra...

In [6]:
# ─────────────────────────────────────────────
# CELL 5 — Save AI Risk Cards To Delta Table
# ─────────────────────────────────────────────

df_risk_cards_spark = spark.createDataFrame(df_risk_cards)

df_risk_cards_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ai_risk_cards")

print("✅ AI Risk Cards saved to Lakehouse")
print(f"   Table: ai_risk_cards")
print(f"   Rows: {df_risk_cards_spark.count()}")
df_risk_cards_spark.show(5, truncate=50)

StatementMeta(, 83995e48-97f3-4db5-9374-620a0f9410ab, 14, Finished, Available, Finished, False)

✅ AI Risk Cards saved to Lakehouse
   Table: ai_risk_cards
   Rows: 50
+----------+-------------+------------------+-------------+-----------+--------------------------------------------------+----------------+
|EmployeeID|         Name|        Department|ML_Risk_Score|  RiskLevel|                                      AI_Risk_Card|    Generated_At|
+----------+-------------+------------------+-------------+-----------+--------------------------------------------------+----------------+
|      4788|  Abdul Basak| IT Infrastructure|         70.5|Medium Risk|Risk Summary:\nAbdul Basak, a Network Engineer ...|2026-07-04 07:49|
|      3198|  Hema Mitter|   Human Resources|         69.3|  High Risk|Risk Summary\nHema Mitter, a dedicated employee...|2026-07-04 07:49|
|      2314|Sudiksha Kari| IT Infrastructure|         68.7|  High Risk|RISK SUMMARY:\nSudiksha Kari, a 12-year veteran...|2026-07-04 07:49|
|      2701|   Faris Sant|Legal & Compliance|         68.2|  High Risk|Risk Summary\nFari